# Pipeline C — Import dependency of business investment in Kosovo

**Question:** How much do business investments in Kosovo depend on imports, and which sectors invest most in the domestic economy?

When a business invests (machinery, equipment, construction), the money either leaves the country (imports) or stays with domestic suppliers. The ATK VAT-declaration report already splits investment by origin, so the question comes from the structure of the data itself.

**Run first** (from the project root):
```
python -m phase_1.run_common_preprocessing
python -m phase_1.pipeline_c.run_pipeline_c
```
This notebook only reads and explains the saved outputs.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/schema.py").exists()), None)
if root is None:
    raise FileNotFoundError("Open the notebook inside the project")
sys.path.insert(0, str(root))
from phase_1.pipeline_c import TABLES, FIGURES, VALIDATION, MEASURE_LABELS, ORIGIN_OF, read_table

status = json.loads(VALIDATION.read_text(encoding="utf-8"))
print("Pipeline C status:", status["status"])
pd.set_option("display.float_format", "{:.3f}".format)

## 1. Feature creation: three origin groups, not two

The six monetary columns are grouped by **origin**. The original Excel headers decide the grouping:

| Source header (ATK) | Group |
|---|---|
| Importet investive me normë 18% / 8% | **import** |
| Blerjet investive vendore me normë 18% / 8% | **domestic** |
| *Blerjet dhe importet* investive pa TVSH | **mixed** |
| *Blerjet dhe importet* investive me TVSH jo të zbritshme | **mixed** |

The two "mixed" columns contain *purchases and imports* together, so their origin cannot be separated. Counting them as imports would overstate import dependency. This matters most in VAT-exempt sectors (education, health, finance), where most purchases fall in the non-deductible VAT column.

Three shares are therefore created:
- `import_share = import / (import + domestic)`: the **main measure**, using only amounts whose origin is known
- `import_share_lower = import / total`: if every mixed euro were domestic
- `import_share_upper = (import + mixed) / total`: if every mixed euro were imported

`import_share` always lies between the two bounds.

In [ ]:
display(pd.DataFrame({"measure": list(MEASURE_LABELS), "label": list(MEASURE_LABELS.values()),
                      "origin": [ORIGIN_OF[m] for m in MEASURE_LABELS]}))

## 2. Aggregation: why shares are not computed per row

Most source rows have no investment at all, or only one origin. A per-row share would be undefined (0/0) or trivially 0% or 100%. Shares are therefore computed on **aggregates** (sector × year, municipality × year, national × year), as a ratio of sums, which weights every euro equally.

In [ ]:
display(read_table("row_level_definedness"))
national = read_table("national_classes")
display(national.query("window == 'reported'")[["year", "months", "import_share", "import_share_lower", "import_share_upper", "mixed_share", "import_dominant"]])

## 3. Feature subset selection

Two criteria, both computed on the source rows:
1. **Variance threshold** on the binary indicator "amount is nonzero": `p(1-p) ≥ 0.95·0.05`. A column that is zero in more than 95% of rows is near-constant.
2. **Value share**: the column must hold at least 1% of all invested value.

The 8% import and 8% domestic columns fail, so they are removed from the multivariate (PCA) feature set. They still stay inside the origin totals, so all totals reconcile with the source.

In [ ]:
display(read_table("feature_selection")[["measure", "origin", "nonzero_rate", "indicator_variance", "value_share", "selected", "reason"]])

## 4. Transformation and dimensionality reduction (PCA)

Matrix: sector × complete year (2019–2025), selected attributes. The amounts are extremely right-skewed, so `log1p` is applied, then a z-score, so that no attribute dominates through its scale.

PCA is computed with the SVD of the standardized matrix. Sign convention: PC1 points towards more investment, and later components point towards the import side.

In [ ]:
display(read_table("transformation_skewness"))
display(read_table("pca_explained_variance"))
display(read_table("pca_loadings")[["label", "origin", "loading_PC1", "loading_PC2", "loading_PC3"]])
display(read_table("pca_origin_contrast"))

In [ ]:
display(Image(filename=str(FIGURES / "c5_pca_biplot.png")))

**Reading:** PC1 is a *size* component, since all loadings are positive (bigger sectors invest more in every category). PC2 does **not** separate imports from domestic purchases. It separates the two *mixed* (no VAT / non-deductible VAT) columns from the pure VAT columns, i.e. the **VAT regime** of the sector. Import vs domestic only appears in PC3 (~5%). So the import/domestic split is a real but minor axis of variation; sector size and VAT treatment dominate.

## 5. Binarization and discretization

- `import_dominant = 1` if `import_share > 0.5`
- `origin_class`: mostly domestic (< 33%), mixed (33–67%), mostly import (> 67%)
- `class_robust`: the class is the same whether the mixed amounts are counted as domestic (lower bound) or as imported (upper bound)
- `relative_class`: tertiles of the rank within each year, i.e. *which sectors rely on imports more than the others*. This stays informative even when almost every sector falls in the same absolute class.

In [ ]:
sectors = read_table("sector_pooled_classes").query("window == 'reported' and sector_eligible")
display(sectors.sort_values("import_share")[["primary_sector", "import_share", "import_share_lower", "import_share_upper",
                                             "import_dominant", "origin_class", "relative_class", "class_robust"]])
display(read_table("class_counts_by_year"))

## 6. Sampling: bootstrap confidence intervals

For each sector, the source rows are resampled with replacement **within each year** (stratified bootstrap, 1,000 replicates, seed 42), and the import share is recomputed. The 2.5–97.5 percentiles give a 95% interval.

`top5_records_import_share` shows how much of a sector's imports come from its 5 largest rows. A high value means the estimate depends on a few large investments.

In [ ]:
boot = read_table("bootstrap_ci").query("sector_eligible and status == 'valid'")
display(boot.sort_values("import_share")[["primary_sector", "records", "import_share", "ci_low", "ci_high", "ci_width",
                                          "top5_records_import_share", "stability"]])

## 7. Results

In [ ]:
for name in ["c1_sector_import_share", "c2_sector_origin_composition", "c3_national_import_share", "c4_municipal_ranking"]:
    display(Image(filename=str(FIGURES / f"{name}.png")))

In [ ]:
display(read_table("municipal_ranking")[["side", "rank", "municipality", "import_share", "identified_total", "origin_class"]])

## 8. Limitations
- About 40% of investment value sits in the two mixed columns, so the true import share is only known within the lower/upper bounds.
- 2026 covers January–June only: compare it through the `jan_jun` window, and leave it out of the PCA.
- Negative adjustments are kept in the totals; aggregates with a negative component get an undefined share instead of a corrected one.
- "Mungon aktiviteti" and "Person Fizik" are not real sectors, so they are excluded from sector comparisons but kept in national and municipal totals.